# GSE280305 — native ScGeo 0.2.1 visualization

Use the **actual plotting API**, not custom reproductions of its figures. The recovered public checkpoint contains 31,605 cells, 3,000 retained HVGs, original PCA50 and Scanorama coordinates. This new descriptive application reads the reviewed D8–D21 geometry/local-geometry tables and reconstitutes the already verified molecular explanation solely to supply the plotting API. D8 is post-irradiation, not an untreated control. No biological sample bootstrap, new integration, trajectory inference or velocity reconstruction is performed.

Set SCGEO_A_H5AD, SCGEO_A_GEOMETRY, SCGEO_A_MOLECULAR, SCGEO_A_LOCAL and SCGEO_A_OUTPUT to explicit local paths; see the companion execution guide. Keep the generated full notebook and source exports local until reviewed. The public previews will be selected **native PlotResult outputs** after execution.

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata as md
import json
import os
import tempfile

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scgeo as sg
from IPython.display import display

def digest(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

source = Path(os.environ["SCGEO_A_H5AD"]).resolve(strict=True)
geometry = Path(os.environ["SCGEO_A_GEOMETRY"]).resolve(strict=True)
molecular = Path(os.environ["SCGEO_A_MOLECULAR"]).resolve(strict=True)
local = Path(os.environ["SCGEO_A_LOCAL"]).resolve(strict=True)
output = Path(os.environ["SCGEO_A_OUTPUT"]).resolve()
expected = "c5d32591abad3d659277a20a5f8681602b55fd849d0bd0e2329c1b4fd6122bee"
assert md.version("scgeo") == "0.2.1"
assert digest(source) == expected
inputs = {
    geometry / "manifest.json": "9f0da2e9f0756e19e1d048d2f89fffdd835c45708aad853ed774cd4f5d84847c",
    geometry / "D8_D21_per_rep_state.csv": "8dc83fb5488f59fdb7aee7cf17e6c30de43517a61ea2394c90575ce640cacf75",
    geometry / "D8_D21_consensus_state.csv": "7f4f6b83a6bb664686467041cd3e3c6910c18cf887602c864216e8faa63f8ed2",
    molecular / "manifest.json": "49c007de0c8ba2b991393531bee564d35baf9daacc90ac62b5453737aa851a69",
    molecular / "D8_D21_summary.csv": "1dcde85577bbc82321eca82cb3c0f67dda202493bd309d7d6dd1b458e9d0432d",
    molecular / "D8_D21_genes.csv.gz": "ae76be1eb55e3a5b66adc630963b2eb7bad7c42136cafe5408d5642a2ae2dfd3",
    local / "representation_summary.csv": "03a3edfe6a3c54a67057ea13f5b5cf9bf60071d6eb4aab7acff005dbe2684c95",
    local / "state_pair_summary.csv": "4c2a021adde44096b88cbc5c5a5939b496757f7a1f521863e451ed4c530de628",
}
assert all(digest(p) == value for p, value in inputs.items()), "Reviewed input mismatch"
for p in (source.parent, geometry, molecular, local):
    assert output != p and p not in output.parents, "Keep outputs separate from inputs"
output.mkdir(parents=True, exist_ok=True)
out = Path(tempfile.mkdtemp(prefix="gse280305-native-", dir=output))
print("Run output:", out)
print("ScGeo:", sg.__version__)
reps = ["X_pca_raw", "X_scanorama"]
states = [str(i) for i in range(14)]
timepoints = ["D8", "D11", "D14", "D21"]
a = ad.read_h5ad(source, backed="r")
try:
    assert a.shape == (31605, 3000)
    assert a.obs_names.is_unique and a.var_names.is_unique
    g = ad.AnnData(X=np.zeros((a.n_obs, 1), dtype=np.float32), obs=a.obs.copy())
    for rep in reps + ["X_umap_raw", "X_umap_scanorama"]:
        g.obsm[rep] = np.asarray(a.obsm[rep]).copy()
        assert np.isfinite(g.obsm[rep]).all()
finally:
    a.file.close()
assert set(g.obs["leiden_raw"].astype(str)) == set(states)
assert set(g.obs["timepoint"].astype(str)) == set(timepoints)
roster = g.obs["timepoint"].astype(str).value_counts().reindex(timepoints)
assert roster.tolist() == [13149, 4513, 5614, 8329]
display(roster)
exports = {}
def save(result, name, scope):
    assert isinstance(result, sg.pl.PlotResult), name
    payload = sg.pl.export_figure(result, out / name, formats=("png", "svg", "pdf"),
        dpi=220, include_data=True, include_metadata=True,
        metadata={"dataset": "GSE280305", "input_sha256": expected,
                  "inference_scope": "descriptive_only", "analysis_scope": scope})
    exports[name] = {"figures": {k: Path(v).name for k, v in payload["figures"].items()},
                     "tables": {k: Path(v).name for k, v in payload["tables"].items()},
                     "metadata": Path(payload["metadata"]).name}
    display(result.fig)
    plt.close(result.fig)


## 1. Matched time-point embeddings

The two coordinate systems are distinct. Within each figure, all four facets share limits and category colors; neither display is evidence of a trajectory.

In [ ]:
for basis in ["umap_raw", "umap_scanorama"]:
    save(sg.pl.embedding_facets(g, basis=basis, facet="timepoint",
        facet_order=timepoints, color="leiden_raw", col_wrap=4,
        shared_limits=True, max_points=12000, point_size=3,
        random_state=1729, return_result=True, show=False,
        title=f"GSE280305 | {basis} | post-irradiation time points"),
        f"01_facets_{basis}", "original stored embeddings; no fitted velocity")


## 2. Native representation diagnostics

Read the previously reviewed D8–D21 state-level estimates and k=15 local-geometry summaries. Table adapters preserve all original values. Local geometry is evaluated on the full recovered roster, not re-estimated for the D8–D21 contrast. A state-level map is appropriate because the previous run did not retain per-cell distortion values.

In [ ]:
per = pd.read_csv(geometry / "D8_D21_per_rep_state.csv", dtype={"node": str})
cons = pd.read_csv(geometry / "D8_D21_consensus_state.csv", dtype={"node": str})
local_rep = pd.read_csv(local / "representation_summary.csv")
local_state = pd.read_csv(local / "state_pair_summary.csv", dtype={"state": str})
assert len(per) == 28 and len(cons) == 14
assert not per.duplicated(["rep", "node"]).any() and cons["node"].is_unique
assert set(per["node"]) == set(cons["node"]) == set(states)
assert per["usable"].all()
assert per["magnitude_ci95_low"].isna().all() and per["magnitude_ci95_high"].isna().all()
assert local_state["k"].eq(15).all()
coverage = pd.crosstab(g.obs["leiden_raw"].astype(str), g.obs["timepoint"].astype(str))
for t, col in [("D8", "n_cells0"), ("D21", "n_cells1")]:
    assert per[col].eq(per["node"].map(coverage[t])).all()
store = {"per_rep_state": per, "consensus_state": cons,
         "representation_diagnostics": local_rep}
g.uns["scgeo"] = {"representation_stability": store,
    "local_geometry_stability": {"params": {"node_key": "leiden_raw", "k_values": [15]},
        "representation_summary": local_rep, "state_pair_summary": local_state}}
save(sg.pl.representation_stability_heatmap(store, state_order=states,
    rep_order=reps, metric="normalized_delta_norm", annotate_consensus=True,
    title="D8 to D21 | Stored representation agreement",
    return_result=True, show=False), "02_representation_heatmap",
    "reviewed state-level D8-D21 effects; full-roster local diagnostic flags")
save(sg.pl.consensus_state_map(g, node_key="leiden_raw", basis="umap_scanorama",
    label_states=True, max_points=12000, random_state=1729,
    title="D8 to D21 | Stored consensus classes",
    return_result=True, show=False), "03_consensus_map",
    "stored consensus classes displayed on original embedding")
for metric in ["neighbor_overlap", "local_shape_distortion"]:
    save(sg.pl.local_distortion_map(g, basis="umap_scanorama",
        node_key="leiden_raw", aggregation="state", metric=metric,
        rep_a=reps[0], rep_b=reps[1], max_points=12000, random_state=1729,
        title=f"All time points | Cluster median {metric}",
        return_result=True, show=False), f"04_{metric}",
        "state-level medians, k=15; not per-cell estimates")


## 3. Native evidence panel and adaptive atlas

Join stored raw-PCA effect and representation-consensus rows by exact cluster identity. Overlay separately calculated full-roster local metrics. No velocity panel is requested because the original projected velocity checkpoint is unavailable. The analysis uses captured-cell comparisons, not independent biological replicates.

In [ ]:
report = per.loc[per["rep"].eq(reps[0]), [
    "node", "n_cells0", "n_cells1", "delta_norm", "normalized_delta_norm"
]].rename(columns={"node": "state"}).copy()
report = report.merge(cons[["node", "consensus_label", "usable_fraction"]].rename(
    columns={"node": "state", "consensus_label": "representation_consensus_label",
             "usable_fraction": "representation_coverage_fraction"}),
    on="state", validate="one_to_one")
for metric, name in [("neighbor_overlap", "median_neighbor_overlap"),
                     ("local_distortion_median", "median_local_shape_distortion")]:
    values = local_state.loc[local_state["metric"].eq(metric), ["state", "median"]]
    assert values["state"].is_unique
    report = report.merge(values.rename(columns={"median": name}), on="state",
                          validate="one_to_one")
assert len(report) == 14
report["group0"], report["group1"] = "D8", "D21"
report["inference_scope"] = "descriptive_only"
report["inference_status"] = "descriptive_only"
report["effect_status"] = "descriptive_only"
report.attrs["provenance"] = {"effect_representation": reps[0], "group0": "D8", "group1": "D21"}
report.to_csv(out / "state_report.csv", index=False)
save(sg.pl.state_evidence_panel(report, effect_metric="normalized_delta_norm",
    state_order=states, max_states=None, show_ci=False,
    title="D8 to D21 | Stored effects and full-roster local diagnostics",
    return_result=True, show=False), "05_evidence_panel",
    "raw-PCA normalized effects; local diagnostics refer to all four time points")
subset = g[g.obs["timepoint"].astype(str).isin(["D8", "D21"])].copy()
save(sg.pl.geometry_dynamics_atlas(subset, basis="umap_scanorama",
    color="leiden_raw", node_key="leiden_raw", condition_key="timepoint",
    group0="D8", group1="D21", panels=("states", "evidence"), report=report,
    effect_metric="normalized_delta_norm", layout="adaptive", max_states=14,
    max_points=10000, random_state=1729,
    title="GSE280305 | Geometry-only atlas; dynamics unavailable",
    return_result=True, show=False), "06_adaptive_atlas",
    "native state and stored-evidence panels; no unsupported dynamics")
del subset


## 4. Native molecular shift curves and connectors

The reviewed archive contains aggregate molecular tables, not the stored cell weights and expression signatures needed by these plotters. Reconstruct the same D8–D21 robust-shift/explanation object in memory, with the original scaled HVG expression and stored PCA loadings. Reject any mismatch with the reviewed gene and displacement tables. This is a rendering prerequisite, not a new discovery analysis. Gene examples are retrospectively selected. Curves use a geometric coordinate, not elapsed time; schematic connectors are not trajectories.

In [ ]:
a = ad.read_h5ad(source, backed="r")
try:
    molecular_data = ad.AnnData(X=np.asarray(a.X[:]), obs=a.obs.copy(), var=a.var.copy())
    for rep in reps + ["X_umap_scanorama"]:
        molecular_data.obsm[rep] = np.asarray(a.obsm[rep]).copy()
    molecular_data.varm["PCs"] = np.asarray(a.varm["PCs"]).copy()
finally:
    a.file.close()
assert molecular_data.obs_names.equals(g.obs_names)
assert molecular_data.var_names.is_unique and np.isfinite(molecular_data.X).all()
sg.tl.robust_shift(molecular_data, rep=reps[0], condition_key="timepoint",
    group0="D8", group1="D21", by="leiden_raw", sample_key=None,
    center="geometric_median", n_boot=0, seed=1729, store_key="reviewed_shift")
explanation = sg.tl.explain_shift(molecular_data, shift_key="reviewed_shift",
    loadings_key="PCs", min_cells=20, min_samples=2, n_boot=0,
    seed=1729, store_key="reviewed_explanation")
old = pd.read_csv(molecular / "D8_D21_summary.csv", dtype={"state": str})
new = explanation["summary"].copy()
new["state"] = new["state"].astype(str)
audit = new.merge(old, on="state", suffixes=("_new", "_reviewed"), validate="one_to_one")
assert len(audit) == len(new) == len(old) == 14
for col in ["status", "n_cells0", "n_cells1", "n_samples0", "n_samples1", "n_boot"]:
    assert audit[f"{col}_new"].eq(audit[f"{col}_reviewed"]).all()
assert new["status"].eq("insufficient_samples").all()
for col in ["delta_norm", "contribution_sum"]:
    np.testing.assert_allclose(audit[f"{col}_new"], audit[f"{col}_reviewed"],
                               rtol=1e-6, atol=1e-5)
old_genes = pd.read_csv(molecular / "D8_D21_genes.csv.gz", dtype={"state": str, "gene": str})
genes = explanation["genes"].copy()
genes["state"] = genes["state"].astype(str)
audit_genes = genes.merge(old_genes, on=["state", "gene"],
    suffixes=("_new", "_reviewed"), validate="one_to_one")
assert len(audit_genes) == len(genes) == len(old_genes) == 42000
for col in ["expression_delta", "axis_loading", "contribution"]:
    np.testing.assert_allclose(audit_genes[f"{col}_new"], audit_genes[f"{col}_reviewed"],
        rtol=1e-6, atol=1e-7)
assert genes[["ci_low", "ci_high", "sign_stability"]].isna().all().all()
print("PASS: matched all 42,000 reviewed gene rows and 14 state explanations")
for state, selected in [("4", ["Nfkbia", "Junb"]), ("9", ["Jchain", "Mzb1"])]:
    save(sg.pl.shift_curves(molecular_data, selected, state=state,
        key="reviewed_explanation", basis="umap_scanorama",
        panels=("curve", "expression", "contribution"), uncertainty="none",
        show_samples=False, max_points=3000, random_state=1729,
        title=f"Cluster {state} | Stored scaled-HVG molecular accounting",
        return_result=True, show=False), f"07_shift_curves_cluster_{state}",
        "verified raw-PCA attribution; pooled descriptive curves; retrospective gene selection")
save(sg.pl.shift_embedding(molecular_data, states=["4", "9"],
    key="reviewed_explanation", basis="umap_scanorama", color="timepoint",
    max_points=10000, random_state=1729,
    title="Clusters 4 and 9 | Schematic condition-center connectors",
    return_result=True, show=False), "08_shift_embedding",
    "stored analysis weights projected to display coordinates; no trajectories")


## 5. Record native plot exports and provenance

The result manifest links the actual package figures to their source tables. Keep the complete run local. After visual inspection, publish selected native SVG/PNG previews and compact aggregate sources in the public gallery. Do not treat this unexecuted source notebook as evidence of successful runtime.

In [ ]:
assert digest(source) == expected
assert all(digest(p) == value for p, value in inputs.items())
files = {p.name: digest(p) for p in sorted(out.iterdir()) if p.is_file()}
direct_url = md.distribution("scgeo").read_text("direct_url.json")
provenance = json.loads(direct_url) if direct_url else {}
manifest = {"status": "complete", "dataset": "GSE280305", "scgeo_version": md.version("scgeo"),
    "scgeo_vcs_commit": provenance.get("vcs_info", {}).get("commit_id"),
    "input_sha256": expected, "source_table_hashes": {str(p.name): v for p, v in inputs.items()
        if p.name != "manifest.json"},
    "n_cells": 31605, "contrast": "D8_D21", "inference_scope": "descriptive_only",
    "native_plot_count": len(exports), "plots": exports, "file_sha256": files,
    "velocity_computed": False}
(out / "native_plot_manifest.json").write_text(json.dumps(manifest, indent=2))
print("PASS: native plots:", len(exports), "|", out)
display(pd.DataFrame([{"plot": name, "formats": len(item["figures"]),
                       "tables": len(item["tables"])} for name, item in exports.items()]))
